In [1]:
import sys
import numpy as np

sys.path.append('../../')
from Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

c:\Users\Menna\AppData\Local\Programs\Python\Python38\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
config = {
    "lib": "tensorflow",
    "partitions": 3,
    "iterations": 3,
    "lr": 0.001,
    "epochs": 20,
    "batch_size": 128,
    "loss": tf.keras.losses.CategoricalCrossentropy(),
    "optimizer": tf.keras.optimizers.legacy.Adam(learning_rate=0.001),
}

In [3]:
def get_train_data():
    return np.load('../../data/MNIST/train_data.npy'), np.load('../../data/MNIST/train_labels.npy')

def get_test_data():
    return np.load('../../data/MNIST/test_data.npy'), np.load('../../data/MNIST/test_labels.npy')

def partition_train_data(X_train, y_train, partitions):
    num_samples = X_train.shape[0]

    # Create an array of indices from 0 to num_samples - 1
    indices = np.arange(num_samples)

    # Shuffle the indices
    np.random.shuffle(indices)

    # Use the shuffled indices to shuffle the datasets
    X_train = X_train[indices]
    y_train = y_train[indices]

    X_train_partitions = []
    y_train_partitions = []

    partition_size = int(len(X_train) / partitions)
    
    for i in range(partitions):
        if i == partitions - 1:
            X_train_partitions.append(X_train[i * partition_size:])
            y_train_partitions.append(y_train[i * partition_size:])
        else:
            X_train_partitions.append(X_train[i * partition_size:(i + 1) * partition_size])
            y_train_partitions.append(y_train[i * partition_size:(i + 1) * partition_size])

    return X_train_partitions, y_train_partitions

In [4]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation('relu'))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation('relu'))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation('softmax'))
    return model

In [5]:
X_train, y_train = get_train_data()
X_train, y_train = partition_train_data(X_train, y_train, config['partitions'])


In [6]:
model = create_model()
rain = Rain(config, model, X_train, y_train)

Rain is initialized
Provisioner created successfully


In [7]:
# rain.setup_vms()

In [8]:
# rain.delete_vms()


In [9]:
model = rain.train_centralized_sync()


divider is running
 divider received: Success recieving the number of workers.
divider is sending data to the coordinator
../../Divider/Algo.py
filepath is opened
 divider received: Success!
../../data/X_train_1.npy
filepath is opened
 divider received: Success!
../../data/y_train_1.npy
filepath is opened
 divider received: Success!


_InactiveRpcError: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNKNOWN
	details = "Exception calling application: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNKNOWN
	details = "Exception iterating requests!"
	debug_error_string = "None"
>"
	debug_error_string = "UNKNOWN:Error received from peer  {created_time:"2023-06-24T12:08:29.288559001+00:00", grpc_status:2, grpc_message:"Exception calling application: <_InactiveRpcError of RPC that terminated with:\n\tstatus = StatusCode.UNKNOWN\n\tdetails = \"Exception iterating requests!\"\n\tdebug_error_string = \"None\"\n>"}"
>

In [ ]:
# X_test, y_test = get_test_data()
# loss, acc = model.evaluate(X_test, y_test, batch_size=config["batch_size"])
# print("\nTest accuracy: %.1f%%" % (100.0 * acc))